In [128]:
import ipywidgets as widgets
import pyarrow as pa
import numpy as np
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
import base64
from io import BytesIO
from PIL import Image
import json

In [192]:
path = "viewfs://hadoop-lt-cluster/home/reco_wl/mpi/luoxinchen/recovlm_dataset_sft/vlm_sft_pure_text"
fs = pa.hdfs.connect(user='mpi')
files = fs.ls(path)
files = [x for x in files if x.endswith("parquet")]

In [193]:
x = np.random.choice(files)
print(x)
df = pq.read_table(x).to_pandas()

viewfs://hadoop-lt-cluster/home/reco_wl/mpi/luoxinchen/recovlm_dataset_sft/vlm_sft_pure_text/rank-1-a34523d8-d59f-11ef-abc5-0894ef9bb154.parquet


In [196]:
def decode_base64_image(base64_string):
    # 解码base64字符串并返回PIL图像对象
    image_data = base64.b64decode(base64_string)
    image = Image.open(BytesIO(image_data))
    return image

def format_messages(messages, images):
    for msg in messages:
        msg_content = f"[{msg['role']}]: "
        for content in msg["content"]:
            if content["type"] == "image":
                if msg_content != "":
                    print(msg_content)
                    msg_content = ""
                name = content["image"]
                image = images[name]
                display(name, image.size, image)
            elif content["type"] == "text":
                msg_content += content["text"]
            elif content["type"] == "video":
                print(content)
        if msg_content != "":
            print(msg_content)
        print("-" * 10)

def format_segments(segments, images):
    for seg in segments:
        if seg["type"] == "text":
            print(seg["text"] + "\n")
        elif seg["type"] == "image":
            name = seg["image"]
            image = images[name]
            display(name, image.size, image)
            

def display_sample(sample):
    source = sample['source']
    images = json.loads(sample.pop('images'))
    messages = json.loads(sample['messages'])
    segments = json.loads(sample['segments'])
    for name, base64_string in images.items():
        image = decode_base64_image(base64_string)
        images[name] = image

    print(f"source={source}, img_nums={len(images)}")
    if segments is not None:
        print("segments sample. Content:")
        print("=" * 10)
        format_segments(segments, images)
    elif messages is not None:
        print("message sample. Content:")
        print("=" * 10)
        format_messages(messages, images)

display_sample(df.sample().iloc[0].to_dict())

source=vlm_sft-language_only, img_nums=0
message sample. Content:
[user]: Can you provide a breakdown of the vitamin and mineral content in kale versus spinach?
----------
[assistant]: Yes, I can provide a breakdown of the vitamin and mineral content in kale and spinach per 100g serving:

KALE:
- Vitamin A: 681% of the Daily Value (DV)
- Vitamin C: 134% of the DV
- Vitamin K: 1339% of the DV
- Folate: 16% of the DV
- Calcium: 9% of the DV
- Iron: 6% of the DV

SPINACH:
- Vitamin A: 187% of the DV
- Vitamin C: 47% of the DV
- Vitamin K: 604% of the DV
- Folate: 15% of the DV
- Calcium: 3% of the DV
- Iron: 15% of the DV

Overall, kale contains more vitamin A, vitamin C, vitamin K, calcium, and folate than spinach. However, spinach contains more iron than kale. Both kale and spinach are nutrient-dense and highly nutritious vegetables, and it is recommended to incorporate both into one's diet for optimal health benefits.
----------
[user]: Wow, I had no idea kale had so much vitamin K! I 

In [ ]:
df